In [1]:
import pandas as pd
import os
import librosa as librs
import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np
import pywt
import time
from pydub import AudioSegment
import wave

In [2]:
def show_split(file_path, y, sr, peak_times):
    file_name = os.path.basename(file_path)
    plt.figure(figsize=(10, 6))
    plt.plot(np.arange(len(y)) / sr, y, label='Onda Audio')
    for peak_time in peak_times:
        plt.axvline(x=peak_time, color='red', linestyle='--', alpha=0.7)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Amplitude')
    plt.title(f'Peaks Found on the Audio Wave: {file_name}')
    plt.legend()
    plt.show()

In [3]:
def split_audio(file_path, peak_times, click_time, output_dir):
    audio = AudioSegment.from_file(file_path)
    t=0;
    while t < (len(peak_times)) :
        start_time = int((peak_times[t]) * 1000)
        if(start_time < 0):
            start_time = 0;
        end_time = int((peak_times[t] + (click_time - 0.1)) * 1000)
        split_audio = audio[start_time:end_time]
        file_name_without_extension = os.path.splitext(os.path.basename(file_path))[0]
        output_filename = os.path.join(output_dir, f"{file_name_without_extension}_split_{t+1}.wav")
        split_audio.export(output_filename, format="wav")
        t+=1;

In [4]:
def peak_detection_and_split(file_path, waveletname, output_dir):
    click_time = 0.3;
    
    # Carica l'audio utilizzando Librosa
    y, sr = librs.load(file_path)

    # Calculation of wavelet transform, noise floor and threshold as twice the noise floor
    coefficients, frequencies = pywt.cwt(y, np.arange(1, 128), waveletname)
    noise_level = np.mean(np.abs(coefficients[:, :int(sr)]))
    threshold = 5 * noise_level
    
    # Trace analysis and memorization of moments when peaks are found.
    i = 0
    peak_times = []
    peak_count = 0
    while i < len(coefficients[0]):
        for j in range(len(coefficients)):
            maybe_peak = abs(coefficients[j, i] - coefficients[j, i - 1])
            # un picco è stato trovato
            if i > 0 and maybe_peak > threshold:
                time_of_peak = librs.samples_to_time(i, sr=sr)
                peak_times.append(time_of_peak)
                i += int(click_time * sr)  # Salta in avanti di x (secondi)
                peak_count += 1             
                break
        i += 1     

    # Call the function to split the audio
    split_audio(file_path, peak_times, click_time, output_dir)
    
    file_name = os.path.basename(file_path)
    
    print(f"The wave is {file_name} using wavelet {waveletname}")
    print(f"Value of threshold: {noise_level}")
    print(f"Number of peaks found: {len(peak_times)}\n")
    
    return peak_times, y, sr;

In [5]:
import pandas as pd
import os
import librosa as librs
import numpy as np
import IPython.display as ipd
import matplotlib.pyplot as plt
import os
import wave
import matplotlib.pylab as pyl
import soundfile as sf
import tensorflow as tf
from tensorflow.keras.layers import Dense,Dropout,Activation,Flatten
from tensorflow.keras.optimizers import Adam
from sklearn import metrics
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import Input, Conv1D, Activation, BatchNormalization, SpatialDropout1D, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from sklearn.metrics import confusion_matrix
from sklearn.utils.multiclass import unique_labels
import seaborn as sns
from sklearn.metrics import classification_report, roc_curve, auc, precision_recall_curve, average_precision_score, confusion_matrix
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import accuracy_score

In [6]:
# import the model

import h5py
import json
from keras.models import model_from_json

# Carica il file HDF5
with h5py.File('tcn_model.h5', 'r') as f:
    # Leggi la struttura del modello (architettura)
    model_config = f.attrs.get('model_config')
    
    # Ricrea il modello dalla configurazione
    tcn_model = model_from_json(model_config)
    
    # Carica i pesi nel modello
    tcn_model.load_weights('tcn_model.h5')

In [7]:
# import the labelencoder

import pickle

# Load del labelencoder
with open('labelencoder.pkl', 'rb') as f:
    labelencoder = pickle.load(f)

In [8]:
# Function to predict password

import os
import re
import resampy

# Define a function to extract the numeric part of the filename
def extract_number(filename):
    return int(re.findall(r'split_(\d+)', filename)[0])

def predict(directory_path, prefix):
    # Initialize an empty list to store predictions
    all_predictions = []

    # Get a list of all files in the directory and sort them based on the numeric part of the filename
    file_list = sorted([f for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f)) and f.startswith(prefix)],
                   key=extract_number)

    for i, file_name in enumerate(file_list, start=1):
        filenamep = os.path.join(directory_path, file_name)
        #print(f"EFFETTUO IL TEST: {i}/{len(file_list)}")
        #print("File selected:", filenamep)
        audio, sample_rate = librs.load(filenamep)
        #extract feature
        mfccs_features = librs.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=36)
        mfccs_scaled_features = np.mean(mfccs_features.T,axis=0)
        mfccs_scaled_features=mfccs_scaled_features.reshape(1,-1)
        
        #prediction
        predicted_label = np.argmax(tcn_model.predict(mfccs_scaled_features), axis=-1)
        prediction_class = labelencoder.inverse_transform(predicted_label) 
        mykey = os.path.splitext(file_name)[0]
        #print("Prediction:", prediction_class[0])
        all_predictions.append(prediction_class[0])
        #print("--------------------------------------------------------------------")

    # Print the list of predictions as a single word
    print("La password riconosciuta è: "+"".join(all_predictions))
    return str(prediction_class[0])